# LTV, Churn Rate, and Customer Lifetime

## 1. The Basic Formula

The common LTV shortcut is:

$$
LTV = \frac{ARPU}{c}
$$

where:

- `LTV` = Customer Lifetime Value
- `ARPU` = Average Revenue Per User
- `c` = Churn Rate

This comes from:

$$
LTV = ARPU \times E[T]
$$

where `E[T]` is the expected customer lifetime.

If churn is constant:

$$
E[T] = \frac{1}{c}
$$

Therefore:

$$
LTV = ARPU \times \frac{1}{c}
$$

and:

$$
\boxed{LTV = \frac{ARPU}{c}}
$$

---

## 2. Example With Units

Suppose:

$$
ARPU = \$50/\text{month}
$$

and:

$$
c = 5\%/\text{month}
$$

Convert 5% to a decimal:

$$
5\% = 0.05
$$

The churn rate is therefore:

$$
c = 0.05/\text{month}
$$

Take the reciprocal:

$$
E[T] = \frac{1}{0.05/\text{month}}
$$

Flipping the fraction gives:

$$
E[T] = 20\text{ months}
$$

Now calculate LTV:

$$
LTV =
\frac{\$50/\text{month}}
{0.05/\text{month}}
$$

The `/month` units cancel:

$$
LTV = \$1,000
$$

Equivalently:

$$
LTV = \$50/\text{month} \times 20\text{ months}
$$

$$
LTV = \$1,000
$$

The important point is:

$$
\frac{1}{0.05/\text{month}}
=
20\text{ months}
$$

The reciprocal of a rate gives a quantity of time.

---

# 3. The Rigorous Statistical Assumption

The relationship:

$$
E[T] = \frac{1}{c}
$$

is not true for every possible churn process.

It is true when the churn hazard is constant.

Let:

$$
T = \text{customer lifetime}
$$

and:

$$
S(t) = P(T > t)
$$

where `S(t)` is the probability that a customer is still active after time `t`.

The hazard rate is:

$$
h(t) =
\lim_{\Delta t \to 0}
\frac{
P(t \leq T < t+\Delta t \mid T \geq t)
}{
\Delta t
}
$$

Think of $h(t)$ as:

> "Given that the customer has survived until now, how likely are they to churn in the next tiny interval?"

If:

$$
h(t) = c
$$

for all `t`, then the hazard is constant.

This produces an exponential lifetime distribution:

$$
S(t) = e^{-ct}
$$

The expected lifetime is:

$$
E[T]
=
\int_0^\infty S(t)\,dt
$$

Therefore:

$$
E[T]
=
\int_0^\infty e^{-ct}\,dt
$$

which gives:

$$
\boxed{E[T] = \frac{1}{c}}
$$

So the rigorous chain is:

$$
\boxed{
\text{Constant hazard}
\Rightarrow
E[T] = \frac{1}{c}
\Rightarrow
LTV = \frac{ARPU}{c}
}
$$

---

# 4. Discrete Monthly Churn

If churn is measured monthly, we can model the process using a geometric distribution.

Suppose every month a surviving customer has probability `c` of churning.

For example:

$$
c = 0.05
$$

Then every month:

- 5% chance of churning
- 95% chance of surviving

The probability of surviving `n` months is:

$$
P(T > n) = (1-c)^n
$$

For example:

$$
P(T > 12) = 0.95^{12}
$$

The expected lifetime is:

$$
E[T] = \frac{1}{c}
$$

Therefore:

$$
E[T] = \frac{1}{0.05}
$$

$$
E[T] = 20\text{ months}
$$

---

# 5. What Does "Constant Churn" Actually Mean?

It means customer age does not affect the probability of churning.

For example:

| Customer Age | Probability of Churning Next Month |
|---|---:|
| 1 month | 5% |
| 6 months | 5% |
| 12 months | 5% |
| 24 months | 5% |
| 60 months | 5% |

A customer who has been around for 5 years has the same probability of churning next month as a brand-new customer.

This is the constant-hazard assumption.

---

# 6. Assumptions Behind LTV = ARPU / Churn

The simple formula:

$$
LTV = \frac{ARPU}{c}
$$

generally assumes:

1. **Constant churn**

   The probability of churning does not depend on customer age.

2. **Stationarity**

   Customer behavior does not substantially change over calendar time.

3. **Customer homogeneity**

   Customers are assumed to have roughly similar churn behavior.

4. **Constant ARPU**

   Revenue per customer per period is approximately constant.

5. **No discounting**

   Future revenue is treated as equally valuable as current revenue.

6. **No expansion or contraction**

   Upgrades, downgrades, cross-sells, etc. are ignored.

7. **Permanent churn**

   Once a customer churns, they do not return.

8. **Accurate churn measurement**

   The observed churn rate is a reasonable estimate of the underlying churn probability.

---

# 7. When the Simple Formula Breaks

Suppose churn depends on customer age:

$$
c(t) =
\begin{cases}
10\% & t \leq 3 \\
5\% & 3 < t \leq 12 \\
2\% & t > 12
\end{cases}
$$

Now:

- New customers have 10% monthly churn.
- Customers aged 4–12 months have 5% monthly churn.
- Customers older than 12 months have 2% monthly churn.

There is no single constant churn rate describing the entire process.

Therefore:

$$
E[T] \neq \frac{1}{\text{arbitrary average churn rate}}
$$

Instead, estimate the survival function:

$$
S(t) = P(T > t)
$$

and calculate:

$$
\boxed{
E[T] = \int_0^\infty S(t)\,dt
}
$$

For discrete monthly data:

$$
\boxed{
E[T] \approx \sum_{t=0}^{\infty} S(t)
}
$$

# 8. Advanced Python Example
## Constant Churn Model

In [2]:
import numpy as np

# Reproducibility
rng = np.random.default_rng(42)

N = 1_000_000
monthly_churn = 0.05
arpu = 50

# Simulate customer lifetimes
lifetimes = []

for _ in range(N):
    lifetime = 0

    while True:
        lifetime += 1

        # Customer has a 5% chance of churning each month
        if rng.random() < monthly_churn:
            break

    lifetimes.append(lifetime)

lifetimes = np.array(lifetimes)

# Empirical average lifetime
observed_lifetime = lifetimes.mean()

# Theoretical average lifetime
theoretical_lifetime = 1 / monthly_churn

# LTV
theoretical_ltv = arpu * theoretical_lifetime
observed_ltv = arpu * observed_lifetime

print("CONSTANT CHURN MODEL")
print("--------------------")
print(f"Theoretical lifetime: {theoretical_lifetime:.2f} months")
print(f"Observed lifetime:    {observed_lifetime:.2f} months")
print(f"Theoretical LTV:      ${theoretical_ltv:,.2f}")
print(f"Observed LTV:         ${observed_ltv:,.2f}")


CONSTANT CHURN MODEL
--------------------
Theoretical lifetime: 20.00 months
Observed lifetime:    20.02 months
Theoretical LTV:      $1,000.00
Observed LTV:         $1,000.97


# 9. Age-Dependent Churn

## Now let's violate the constant-churn assumption.

In [3]:
def churn_probability(age):
    """
    Churn probability depends on customer age.
    """

    if age <= 3:
        return 0.10      # 10% for first 3 months
    elif age <= 12:
        return 0.05      # 5% for months 4-12
    else:
        return 0.02      # 2% thereafter

lifetimes_variable = []

for _ in range(N):
    age = 0

    while True:
        age += 1

        if rng.random() < churn_probability(age):
            break

    lifetimes_variable.append(age)

lifetimes_variable = np.array(lifetimes_variable)

actual_lifetime = lifetimes_variable.mean()

print("AGE-DEPENDENT CHURN MODEL")
print("-------------------------")
print(f"Actual average lifetime: {actual_lifetime:.2f} months")

AGE-DEPENDENT CHURN MODEL
-------------------------
Actual average lifetime: 30.96 months


Here:

$$
c(t) =
\begin{cases}
10\% & t \leq 3 \\
5\% & 3 < t \leq 12 \\
2\% & t > 12
\end{cases}
$$

 Because the hazard changes with age, the simple:

$$
\frac{1}{c}
$$

shortcut is no longer generally valid.

# 10\. Estimate Lifetime From the Survival Curve

## Instead of assuming constant churn, we can estimate the survival function directly.

In [4]:
max_month = lifetimes_variable.max()

survival = np.array([
    np.mean(lifetimes_variable > month)
    for month in range(max_month + 1)
])

# Discrete approximation to expected lifetime
estimated_lifetime = survival.sum()

print("SURVIVAL ANALYSIS")
print("-----------------")
print(f"Estimated lifetime: {estimated_lifetime:.2f} months")
print(f"Direct sample mean: {actual_lifetime:.2f} months")



SURVIVAL ANALYSIS
-----------------
Estimated lifetime: 30.96 months
Direct sample mean: 30.96 months


The estimated lifetime from the survival curve should be very close to the direct sample mean.

---

# 11. Conceptual Hierarchy

## The simple model is:

$$
\boxed{
\text{Constant churn}
\Rightarrow
E[T] = \frac{1}{c}
\Rightarrow
LTV = \frac{ARPU}{c}
}
$$

The more general model is:

$$
\boxed{
\text{Age-dependent churn}
\Rightarrow
S(t)
\Rightarrow
E[T] = \int_0^\infty S(t)\,dt
\Rightarrow
LTV = ARPU \times E[T]
}
$$

 Therefore:

 > `LTV = ARPU / churn` is not a fundamental law of LTV.

 It is a convenient special case that follows from a constant-churn model.

 When churn varies with customer age, survival analysis provides the more rigorous framework.